# Run ARCH-plan python package

## Before running

1. Install ARCH once by running `./run.sh` from the root of the project.
2. Select the `.venv` environment (created by `run.sh`) as the kernel of this notebook.

## Run the REST API and web app

The cell below starts the server and keeps running until it is interrupted (stop button of the cell).

While it runs:
- web app: http://localhost:8001
- API documentation: http://localhost:8001/docs

In [ ]:
import subprocess
import sys
from pathlib import Path


def find_project_root(start: Path = Path.cwd()) -> Path:
    """Return the root folder of the project (the one containing pyproject.toml).

    Args:
        start: Folder from which the search starts, moving up through its parents.
            Defaults to the current working directory.

    Returns:
        Path of the project root folder.

    Raises:
        FileNotFoundError: If no parent folder contains pyproject.toml.
    """
    for folder in [start, *start.parents]:
        if (folder / "pyproject.toml").is_file():
            return folder
    raise FileNotFoundError(f"pyproject.toml not found in {start} or its parent folders")


def run_server(port: int = 8001, reload: bool = True) -> None:
    """Start the ARCH REST API and web app, printing the server output in this cell.

    The server runs from the src/ folder of the project. This function blocks until
    the cell is interrupted, and then stops the server.

    Args:
        port: Port where the REST API and web app are served. Defaults to 8001.
        reload: If True, run in development mode, restarting the server when the
            code changes. If False, run in production mode. Defaults to True.

    Returns:
        None.
    """
    mode: str = "dev" if reload else "run"
    command: list[str] = [sys.executable, "-m", "fastapi", mode, "arch_plan/server.py", "--port", str(port)]
    server = subprocess.Popen(
        command,
        cwd=find_project_root() / "src",
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    try:
        for line in server.stdout:
            print(line, end="")
    except KeyboardInterrupt:
        print("Stopping the server...")
    finally:
        server.terminate()
        server.wait()


run_server(port=8001)